In [1]:
import os
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import label_binarize
from copy import deepcopy
from sklearn.utils.class_weight import compute_class_weight

[HAMI-core Msg(84826:139861655805248:libvgpu.c:837)]: Initializing.....
[HAMI-core Msg(84826:139861655805248:libvgpu.c:856)]: Initialized


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [4]:
def model_fn():
    in_channels = client_graphs[0].num_node_features
    out_channels = client_graphs[0].y.max().item() + 1
    hidden_channels = 64
    return GraphSAGE(in_channels, hidden_channels, out_channels).to(device)

In [5]:
def local_train(model, data, optimizer, loss_fn, local_epochs=1):
    model.train()
    for _ in range(local_epochs):
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
    return model.state_dict()

In [6]:
def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out = model(data)
        loss = loss_fn(out[mask], data.y[mask]).item()
        pred = out.argmax(dim=1)
        acc = (pred[mask] == data.y[mask]).sum() / mask.sum()
    return loss, acc.item()

In [7]:
def evaluate_on_test(model, data):
    model.eval()
    with torch.no_grad():
        out = model(data)
        if out.isnan().any():
            print("Model output contains NaNs")
            return float("nan")
        if data.test_mask.sum() == 0:
            print("Test mask is empty")
            return float("nan")
        pred = out.argmax(dim=1)
        acc = (pred[data.test_mask] == data.y[data.test_mask]).sum() / data.test_mask.sum()
    return acc.item()

In [8]:
def predict_on_test(model, data):
    model.eval()
    with torch.no_grad():
        logits = model(data)
        test_logits = logits[data.test_mask]
        y_true = data.y[data.test_mask].cpu().numpy()
        y_pred = test_logits.argmax(dim=1).cpu().numpy()
        y_score = torch.softmax(test_logits, dim=1).cpu().numpy()
    return y_true, y_pred, y_score

In [9]:
# ─────────────────────────────────────────────
# FEDAVG
# ─────────────────────────────────────────────
from collections import defaultdict
from copy import deepcopy
import torch
from sklearn.cluster import KMeans


def fedavg(state_dicts):
    avg_state = deepcopy(state_dicts[0])
    for key in avg_state:
        for i in range(1, len(state_dicts)):
            avg_state[key] += state_dicts[i][key]
        avg_state[key] /= len(state_dicts)
    return avg_state

# ─────────────────────────────────────────────
# FEDAVGM
# ─────────────────────────────────────────────
def fedavgm(updates, global_weights, momentum_buffer, beta=0.9):
    delta = {}
    for key in global_weights:
        delta[key] = sum(update[key] - global_weights[key] for update in updates) / len(updates)
        momentum_buffer[key] = beta * momentum_buffer.get(key, torch.zeros_like(delta[key])) + delta[key]
    new_global_weights = {k: global_weights[k] + momentum_buffer[k] for k in global_weights}
    return new_global_weights, momentum_buffer

# ─────────────────────────────────────────────
# FEDPROX - Local Training
# ─────────────────────────────────────────────
def local_train_prox(model, global_weights, data, optimizer, loss_fn, mu=0.01, local_epochs=1):
    model.train()
    for _ in range(local_epochs):
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
        prox_term = sum(((param - global_weights[name]) ** 2).sum() 
                        for name, param in model.named_parameters())
        loss += (mu / 2) * prox_term
        loss.backward()
        optimizer.step()
    return model.state_dict()

# ─────────────────────────────────────────────
# FEDNOVA
# ─────────────────────────────────────────────
def fednova(updates, local_steps):
    total_steps = sum(local_steps)
    weighted_updates = {}
    for key in updates[0]:
        weighted_updates[key] = sum(local_steps[i] * updates[i][key] for i in range(len(updates))) / total_steps
    return weighted_updates

# ─────────────────────────────────────────────
# FEDCURV
# ─────────────────────────────────────────────
def apply_fedcurv_aggregation(global_weights, updates, fim_list, lambda_=0.1):
    agg_weights = {}
    for k in global_weights:
        numerator = sum(fim_list[i][k] * updates[i][k] for i in range(len(updates)))
        denominator = sum(fim_list[i][k] + 1e-8 for i in range(len(updates)))
        agg_weights[k] = (1 - lambda_) * global_weights[k] + lambda_ * (numerator / denominator)
    return agg_weights

def estimate_fim(model, data, loss_fn):
    fim = {}
    model.eval()
    out = model(data)
    loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
    grads = torch.autograd.grad(loss, model.parameters(), create_graph=False)
    for (name, param), grad in zip(model.named_parameters(), grads):
        fim[name] = grad.detach() ** 2
    return fim

# ─────────────────────────────────────────────
# CLUSTERED FL
# ─────────────────────────────────────────────
def cluster_clients(model_updates, n_clusters=2):
    flat_updates = [torch.cat([p.flatten() for p in update.values()]) for update in model_updates]
    X = torch.stack(flat_updates).cpu().numpy()
    labels = KMeans(n_clusters=n_clusters, random_state=0).fit_predict(X)
    return labels

def aggregate_by_cluster(updates, labels):
    cluster_groups = defaultdict(list)
    for i, label in enumerate(labels):
        cluster_groups[label].append(updates[i])

    cluster_avgs = [fedavg(group) for group in cluster_groups.values()]
    final_global = {}
    for k in cluster_avgs[0]:
        final_global[k] = sum(cluster[k] for cluster in cluster_avgs) / len(cluster_avgs)
    return final_global

# ─────────────────────────────────────────────
# ATTENTION-BASED AGGREGATION
# ─────────────────────────────────────────────
def attention_aggregation(updates, global_weights):
    weights = []
    for update in updates:
        diff = torch.cat([((update[k] - global_weights[k])**2).flatten() for k in update])
        weights.append(torch.exp(-torch.norm(diff)))
    weights = torch.tensor(weights)
    weights = weights / weights.sum()

    agg = {}
    for k in updates[0]:
        agg[k] = sum(weights[i] * updates[i][k] for i in range(len(updates)))
    return agg

In [10]:
data_path = "federated_data"
client_ids = ["client1", "client2", "client3", "client4", "client5"]
client_graphs = []

In [11]:
for cid in client_ids:
    path = os.path.join(data_path, cid, "knn_k5_los_data.pt")
    graph = torch.load(path).to(device)
    client_graphs.append(graph)

/tmp/ipykernel_84826/2947992221.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  graph = torch.load(path).to(device)


In [12]:
test_graph = torch.load(os.path.join(data_path, "test", "knn_k5_los_data.pt")).to(device)

/tmp/ipykernel_84826/1996089326.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_graph = torch.load(os.path.join(data_path, "test", "knn_k5_los_data.pt")).to(device)

In [13]:
 # loss_fn = torch.nn.CrossEntropyLoss()

train_labels = []
for data in client_graphs:
    train_labels.append(data.y[data.train_mask])
all_train_labels = torch.cat(train_labels).cpu().numpy()

classes = np.unique(all_train_labels)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=all_train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)


In [14]:

def run_federated_training_experiment(aggregation_strategy, save_dir="GraphSAGE_FL_Results"):

    num_rounds = 100
    local_epochs = 10
    lr = 1e-4
    mu = 0.01     
    beta = 0.9      

    output_path = os.path.join(save_dir, aggregation_strategy)
    os.makedirs(output_path, exist_ok=True)

    # Create result directory
    output_path = os.path.join(save_dir, aggregation_strategy)
    os.makedirs(output_path, exist_ok=True)

    # Reinitialize global model
    global_model = model_fn().to(device)
    global_weights = deepcopy(global_model.state_dict())
    momentum_buffer = {}

    val_acc_per_round = []
    val_loss_per_round = []
    test_acc_list = []

    print(f"\n=== Training with Aggregation Strategy: {aggregation_strategy} ===\n")

    if aggregation_strategy == "fedcurv":
        fim_list = [None for _ in client_graphs]

    for rnd in range(1, num_rounds + 1):
        local_states = []
        val_accuracies = []
        val_losses = []
        local_steps = []

        print(f"\n--- Round {rnd} ---")

        for i, data in enumerate(client_graphs):
            local_model = model_fn().to(device)
            local_model.load_state_dict(global_weights)
            optimizer = torch.optim.Adam(local_model.parameters(), lr=lr, weight_decay=5e-4)

            if aggregation_strategy == "fedprox":
                state_dict = local_train_prox(local_model, global_weights, data, optimizer, loss_fn, mu, local_epochs)

            else:
                state_dict = local_train(local_model, data, optimizer, loss_fn, local_epochs)

            local_states.append(state_dict)
            local_steps.append(local_epochs)

            if aggregation_strategy == "fedcurv":
                fim_list[i] = estimate_fim(local_model, data, loss_fn)

            val_loss, val_acc = evaluate(local_model, data, data.val_mask)
            val_losses.append(val_loss)
            val_accuracies.append(val_acc)
            print(f"  [Client {i+1}] Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Aggregation
        if aggregation_strategy == "fedavg":
            global_weights = fedavg(local_states)

        elif aggregation_strategy == "fedavgm":
            global_weights, momentum_buffer = fedavgm(local_states, global_weights, momentum_buffer, beta)

        elif aggregation_strategy == "fedprox":
            global_weights = fedavg(local_states)  # Same as FedAvg

        elif aggregation_strategy == "fednova":
            global_weights = fednova(local_states, local_steps)

        elif aggregation_strategy == "attention":
            global_weights = attention_aggregation(local_states, global_weights)

        elif aggregation_strategy == "fedcurv":
            global_weights = apply_fedcurv_aggregation(global_weights, local_states, fim_list)

        elif aggregation_strategy == "clustered":
            labels = cluster_clients(local_states)
            global_weights = aggregate_by_cluster(local_states, labels)  

        else:
            raise ValueError(f"Unsupported aggregation strategy: {aggregation_strategy}")

        global_model.load_state_dict(global_weights)

        # Evaluate on test
        test_acc = evaluate_on_test(global_model, test_graph)
        test_acc_list.append(test_acc)

        avg_val_acc = sum(val_accuracies) / len(val_accuracies)
        avg_val_loss = sum(val_losses) / len(val_losses)

        val_acc_per_round.append(avg_val_acc)
        val_loss_per_round.append(avg_val_loss)

        print(f"[Round {rnd:02d}] Avg Val Loss: {avg_val_loss:.4f} | Avg Val Acc: {avg_val_acc:.4f}")
        print(f"[Round {rnd:02d}] Global Test Accuracy: {test_acc:.4f}")

    # Save final global model
    torch.save(global_model.state_dict(), os.path.join(output_path, "global_model.pth"))

    # Plot and save validation accuracy curve
    rounds = list(range(1, num_rounds + 1))

    plt.figure(figsize=(8, 5))
    plt.plot(rounds, val_loss_per_round, label='Validation Loss', color='orange', linewidth=2)
    plt.plot(rounds, val_acc_per_round, label='Validation Accuracy', color='blue', linewidth=2)
    
    plt.xlabel("Federated Rounds")
    plt.ylabel("Metric Value")
    plt.title("Validation Loss and Accuracy over Federated Rounds")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, "val_accuracy_curve.png"))
    plt.close()

    # --- Evaluate on test set (classification report + AUC + Confusion Matrix) ---
    y_true, y_pred, y_score = predict_on_test(global_model, test_graph)

    class_names = ['Short', 'Medium', 'Long']  # update if needed

    # Classification Report
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    with open(os.path.join(output_path, "classification_report.txt"), "w") as f:
        f.write(classification_report(y_true, y_pred, target_names=class_names))

    # Classification Heatmap
    report_dict = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
    )
    df = pd.DataFrame(report_dict).T
    df = df.drop(columns=["support"])
    
    
    plt.figure(figsize=(10, len(df) * 0.6))
    sns.heatmap(df, annot=True, cmap="GnBu", fmt=".2f",
            annot_kws={"size": 14}, 
                cbar_kws={"shrink": 0.6})
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.title("Classification Report Heatmap", fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, "classification_heatmap.png"))
    plt.close()

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                         xticklabels=class_names, yticklabels=class_names,
                         annot_kws={"size": 14})
    # disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Short", "Medium", "Long"])
    # disp.plot(cmap='Blues', values_format='d', ax=ax)
    
    plt.title("Confusion Matrix (Global Model on Test Set)", fontsize=14)
    ax.set_xlabel("Predicted Label", fontsize=14)
    ax.set_ylabel("True Label", fontsize=14)
    ax.tick_params(axis='both', labelsize=14)
    
    plt.tight_layout()    
    plt.savefig(os.path.join(output_path, "confusion_matrix.png"))
    plt.close()

    # AUC scores
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    macro_auc = roc_auc_score(y_true_bin, y_score, average="macro")
    weighted_auc = roc_auc_score(y_true_bin, y_score, average="weighted")
    with open(os.path.join(output_path, "auc_scores.txt"), "w") as f:
        f.write(f"Macro AUC: {macro_auc:.4f}\nWeighted AUC: {weighted_auc:.4f}")

    print(f"\nResults saved in: {output_path}")

    # ROC Curves
    n_classes = len(class_names)
    
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    plt.figure(figsize=(8, 6))
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")
    
    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Multiclass ROC Curve")
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, "ROC Curves.png"))
    plt.close()

In [15]:
def run_all_federated_strategies(strategies=None, base_save_dir="results"):
    if strategies is None:
        strategies = ["fedavg", "fedavgm", "fedprox", "fednova", "fedcurv", "clustered", "attention"]

    for strategy in strategies:
        try:
            run_federated_training_experiment(strategy, save_dir=base_save_dir)
        except Exception as e:
            print(f"Failed to run strategy '{strategy}': {str(e)}")

In [16]:
run_all_federated_strategies()


=== Training with Aggregation Strategy: fedavg ===


--- Round 1 ---
  [Client 1] Val Loss: 1.1932 | Val Acc: 0.4107
  [Client 2] Val Loss: 1.2060 | Val Acc: 0.3899
  [Client 3] Val Loss: 1.2549 | Val Acc: 0.3580
  [Client 4] Val Loss: 1.2297 | Val Acc: 0.3432
  [Client 5] Val Loss: 8.3513 | Val Acc: 0.3758
[Round 01] Avg Val Loss: 2.6470 | Avg Val Acc: 0.3755
[Round 01] Global Test Accuracy: 0.3897

--- Round 2 ---
  [Client 1] Val Loss: 1.1119 | Val Acc: 0.4522
  [Client 2] Val Loss: 1.1183 | Val Acc: 0.3892
  [Client 3] Val Loss: 1.1380 | Val Acc: 0.4018
  [Client 4] Val Loss: 1.1184 | Val Acc: 0.3803
  [Client 5] Val Loss: 9.7806 | Val Acc: 0.4003
[Round 02] Avg Val Loss: 2.8535 | Avg Val Acc: 0.4047
[Round 02] Global Test Accuracy: 0.4061

--- Round 3 ---
  [Client 1] Val Loss: 1.0742 | Val Acc: 0.4981
  [Client 2] Val Loss: 1.0737 | Val Acc: 0.4855
  [Client 3] Val Loss: 1.0830 | Val Acc: 0.4596
  [Client 4] Val Loss: 1.0750 | Val Acc: 0.5130
  [Client 5] Val Loss: 7.8931 | Val 